# Supervised Anomaly Detection with Activity Recognition as Auxiliary Task

## Motivation

The baseline `classification.ipynb` trains a CNN encoder on a single operation (OP05).  
Here we introduce **multi-task learning**: the same encoder is trained simultaneously on

| Task | Target | Loss weight |
|------|--------|-------------|
| **Primary** – anomaly detection | good (0) / bad (1) | 1.0 |
| **Auxiliary** – operation recognition | OP index (0-11) | `aux_weight` |

Classifying which machining operation is active forces the encoder to capture
operation-specific vibration characteristics (tool type, spindle speed, cutting depth).  
These richer representations also generalise better to the anomaly detection task.

Training uses **all 12 operations** that have labelled good/bad states.  
Evaluation reports results on:
1. **OP05 test set** — identical split to the baseline for a direct comparison.  
2. **All-operations test set** — shows generalisation across the full machine lifecycle.

In [1]:
!git clone https://github.com/pydamavand/damavand
!pip install -r damavand/requirements.txt
!git clone https://github.com/boschresearch/CNC_Machining.git

fatal: destination path 'damavand' already exists and is not an empty directory.
fatal: destination path 'CNC_Machining' already exists and is not an empty directory.


In [2]:
import os
import h5py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from itertools import combinations

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from damavand.damavand.utils import splitter, z_score_scaler

print("Imports OK")

Imports OK


## Dataset

In [3]:
class Dataset:
    def __init__(
        self,
        dir: str = "/content/CNC_Machining/data/",
        machine: str = "M01",
        operations: list = [f"{i:02d}" for i in range(15)],
        channels: list = ["0", "1", "2"],
    ):
        self.dir = dir
        self.machine = machine
        self.operations = operations
        self.channels = channels
        self.data = {f"OP{op}": {c: [] for c in channels} for op in self.operations}

    def mine(self, win_len: int = 500, hop_len: int = 500):
        for op in self.operations:
            op_path = os.path.join(self.dir, self.machine, f"OP{op}")
            for state in os.listdir(op_path):
                state_path = os.path.join(op_path, state)
                for file in os.listdir(state_path):
                    fpath = os.path.join(state_path, file)
                    with h5py.File(fpath, "r") as f:
                        print(fpath, " --- ", f["vibration_data"].shape)
                        for channel in self.channels:
                            temp_df = splitter(
                                f["vibration_data"][:, int(channel)], win_len, hop_len
                            )
                            temp_df["machine"]   = self.machine
                            temp_df["operation"] = f"OP{op}"
                            temp_df["state"]     = state
                            temp_df["file"]      = file
                            self.data[f"OP{op}"][channel].append(temp_df)
        return self.data

print("Dataset class defined.")

Dataset class defined.


## Model Classes

```
Input (1×1000)
    │
    ▼
VibrationFeatureExtractor   ← shared CNN encoder (320-d)
    │
    ├─── anomaly_head  → 2 logits  (good / bad)       [primary task]
    │
    └─── op_head       → 12 logits (OP00…OP14 subset) [auxiliary task]
```

Joint loss: `L = L_anomaly + aux_weight × L_op`

In [4]:
class VibrationFeatureExtractor(nn.Module):
    def __init__(self):
        super(VibrationFeatureExtractor, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=32, kernel_size=64, stride=4, padding=32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=2),
            nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(5),
        )
        self.feature_dim = 64 * 5  # 320

    def forward(self, x):
        x = self.features(x)
        return x.view(x.size(0), -1)


class ContrastiveModel(nn.Module):
    def __init__(self, encoder, embedding_dim=64, num_classes=2):
        super(ContrastiveModel, self).__init__()
        self.encoder = encoder
        self.projection_head = nn.Sequential(
            nn.Linear(self.encoder.feature_dim, 128),
            nn.ReLU(),
            nn.Linear(128, embedding_dim),
        )
        self.classifier_head = nn.Linear(self.encoder.feature_dim, num_classes)
        self.mode = "contrastive"

    def set_mode(self, mode):
        assert mode in ["contrastive", "classify"]
        self.mode = mode

    def forward(self, x):
        features = self.encoder(x)
        if self.mode == "contrastive":
            return self.projection_head(features)
        return self.classifier_head(features)


class MultiTaskModel(nn.Module):
    """
    Shared CNN encoder with two output heads for multi-task learning.
    Primary task  — anomaly_head : binary good/bad state detection.
    Auxiliary task — op_head     : CNC operation classification.
    """
    def __init__(self, encoder, num_ops: int = 12):
        super(MultiTaskModel, self).__init__()
        self.encoder = encoder
        self.anomaly_head = nn.Linear(encoder.feature_dim, 2)
        self.op_head = nn.Linear(encoder.feature_dim, num_ops)

    def forward(self, x):
        features = self.encoder(x)
        return self.anomaly_head(features), self.op_head(features)

    def predict_anomaly(self, x):
        return self.anomaly_head(self.encoder(x))


print("Model classes defined.")

Model classes defined.


## Utility Functions

In [5]:
class MultiTaskDataset(torch.utils.data.Dataset):
    """Yields (signal, anomaly_label, op_label) triplets."""
    def __init__(self, x_data, y_anomaly, y_op):
        self.x = torch.tensor(np.asarray(x_data), dtype=torch.float32).unsqueeze(1)
        self.y_anomaly = torch.tensor(
            y_anomaly.to_numpy() if hasattr(y_anomaly, "to_numpy") else np.asarray(y_anomaly),
            dtype=torch.long,
        )
        self.y_op = torch.tensor(
            y_op.to_numpy() if hasattr(y_op, "to_numpy") else np.asarray(y_op),
            dtype=torch.long,
        )

    def __len__(self):
        return len(self.y_anomaly)

    def __getitem__(self, idx):
        return self.x[idx], self.y_anomaly[idx], self.y_op[idx]


class ContrastiveDataset(torch.utils.data.Dataset):
    def __init__(self, x1, x2, y):
        self.x1, self.x2, self.y = x1, x2, y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x1[idx], self.x2[idx], self.y[idx]


def contrastive_loss(z1, z2, label, margin=1.0):
    dist = torch.nn.functional.pairwise_distance(z1, z2)
    loss_pos = label * torch.pow(dist, 2)
    loss_neg = (1 - label) * torch.pow(torch.clamp(margin - dist, min=0.0), 2)
    return torch.mean(loss_pos + loss_neg)


def create_balanced_contrastive_pairs(x_train, y_train):
    y_arr = y_train.to_numpy() if hasattr(y_train, "to_numpy") else np.asarray(y_train)
    x_arr = np.asarray(x_train)
    good_idx = np.where(y_arr == 0)[0]
    bad_idx  = np.where(y_arr == 1)[0]
    np.random.seed(42)
    good_idx = np.random.choice(good_idx, size=len(bad_idx), replace=False)
    idx = np.concatenate([good_idx, bad_idx])
    x_bal, y_bal = x_arr[idx], y_arr[idx]
    pairs = list(combinations(range(len(y_bal)), 2))
    x1_list, x2_list, labels = [], [], []
    for i, j in pairs:
        x1_list.append(x_bal[i])
        x2_list.append(x_bal[j])
        labels.append(1.0 if y_bal[i] == y_bal[j] else 0.0)
    X1 = torch.tensor(np.array(x1_list), dtype=torch.float32).unsqueeze(1)
    X2 = torch.tensor(np.array(x2_list), dtype=torch.float32).unsqueeze(1)
    Y  = torch.tensor(np.array(labels),  dtype=torch.float32)
    return X1, X2, Y


def evaluate_multitask_model(model, x, y, target_names=["good", "bad"], config=None, export_path=None):
    """Evaluates the anomaly detection head of MultiTaskModel."""
    model.eval()
    x_tensor = torch.tensor(np.asarray(x), dtype=torch.float32).unsqueeze(1)
    y_array  = y.to_numpy() if hasattr(y, "to_numpy") else np.asarray(y)
    preds = []
    with torch.no_grad():
        for start in range(0, len(x_tensor), 64):
            logits = model.predict_anomaly(x_tensor[start:start+64])
            _, p = torch.max(logits, 1)
            preds.extend(p.cpu().numpy())
    preds = np.array(preds)
    report = classification_report(y_array, preds, target_names=target_names)
    cm = confusion_matrix(y_array, preds)
    config_str = "\n".join([f"  {k}: {v}" for k, v in config.items()]) if config else "  None"
    output = "\n".join([
        "\n============= Experiment configuration =============",
        "config:", config_str,
        "\n================ Evaluation Metrics ================",
        report,
        "Confusion Matrix:",
        f"True {target_names[0].capitalize()}: {cm[0,0]} | False {target_names[1].capitalize()}: {cm[0,1]}",
        f"False {target_names[0].capitalize()}: {cm[1,0]} | True {target_names[1].capitalize()}: {cm[1,1]}\n",
    ])
    print(output)
    if export_path:
        with open(export_path, "a") as f:
            f.write(output + "\n" + "=" * 50 + "\n")
        print(f"--> Saved to: {export_path}")
    return {"predictions": preds, "true_labels": y_array, "confusion_matrix": cm}


print("Utility functions defined.")

Utility functions defined.


## Configuration

Key new parameter: **`aux_weight`** — scales the operation-recognition loss relative
to the anomaly-detection loss. A value of 0.5 means the auxiliary task contributes
half as much gradient as the primary task.

In [6]:
config = {
    "do_contrastive_pretraining": True,
    "downsample_majority_class": True,
    "aux_weight": 0.5,
    "contrastive_epochs": 10,
    "classification_epochs": 15,
    "contrastive_lr": 0.001,
    "classification_lr": 0.0005,
    "batch_size_contrastive": 128,
    "batch_size_classify": 32,
}

## 1. Load Dataset — All 12 Operations

In [7]:
OPERATIONS = ["00", "01", "03", "04", "05", "06", "07", "08", "10", "11", "12", "14"]
OP_TO_IDX  = {f"OP{op}": i for i, op in enumerate(OPERATIONS)}

dataset = Dataset(operations=OPERATIONS)
data    = dataset.mine(win_len=1000, hop_len=1000)
print("Dataset loaded.")

/content/CNC_Machining/data/M01/OP00/good/M01_Aug_2019_OP00_013.h5  ---  (268288, 3)
/content/CNC_Machining/data/M01/OP00/good/M01_Aug_2019_OP00_001.h5  ---  (268288, 3)
/content/CNC_Machining/data/M01/OP00/good/M01_Aug_2019_OP00_007.h5  ---  (269312, 3)
/content/CNC_Machining/data/M01/OP00/good/M01_Aug_2019_OP00_009.h5  ---  (263168, 3)
/content/CNC_Machining/data/M01/OP00/good/M01_Aug_2019_OP00_000.h5  ---  (268288, 3)
/content/CNC_Machining/data/M01/OP00/good/M01_Feb_2021_OP00_006.h5  ---  (268288, 3)
/content/CNC_Machining/data/M01/OP00/good/M01_Aug_2019_OP00_012.h5  ---  (269312, 3)
/content/CNC_Machining/data/M01/OP00/good/M01_Feb_2021_OP00_003.h5  ---  (269312, 3)
/content/CNC_Machining/data/M01/OP00/good/M01_Aug_2019_OP00_011.h5  ---  (268288, 3)
/content/CNC_Machining/data/M01/OP00/good/M01_Aug_2019_OP00_005.h5  ---  (268288, 3)
/content/CNC_Machining/data/M01/OP00/good/M01_Aug_2019_OP00_008.h5  ---  (268288, 3)
/content/CNC_Machining/data/M01/OP00/good/M01_Feb_2019_OP00_004.h

## 2. Per-Operation Train / Test Splits

Each operation is split independently with `test_size=0.4, random_state=42`,
so the **OP05 test set is identical to the one used in the baseline** —
enabling a direct, apples-to-apples performance comparison.

In [8]:
train_dfs, test_dfs = [], []
op05_test_df = None

for op_id in OPERATIONS:
    df_op = pd.concat(data[f"OP{op_id}"]["0"])
    df_op_train, df_op_test = train_test_split(df_op, test_size=0.4, random_state=42)
    train_dfs.append(df_op_train)
    test_dfs.append(df_op_test)
    if op_id == "05":
        op05_test_df = df_op_test.copy()

df_train = pd.concat(train_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
df_test  = pd.concat(test_dfs).reset_index(drop=True)

for df in [df_train, df_test, op05_test_df]:
    df["state_encoded"] = df["state"].map({"good": 0, "bad": 1})
    df["op_encoded"]    = df["operation"].map(OP_TO_IDX)

print(f"Training set  : {len(df_train):,} windows across {len(OPERATIONS)} operations")
print(f"Test set (all): {len(df_test):,} windows")
print(f"OP05 test set : {len(op05_test_df):,} windows  (← same as baseline)")
print("\nTraining class distribution:")
print(df_train["state_encoded"].value_counts())

Training set  : 24,743 windows across 12 operations
Test set (all): 16,504 windows
OP05 test set : 622 windows  (← same as baseline)

Training class distribution:
state_encoded
0    23169
1     1574
Name: count, dtype: int64


## 3. Optional Majority-Class Downsampling

We balance each operation's training slice independently before combining
them, so no single operation drowns out the minority-class signal.

In [9]:
if config["downsample_majority_class"]:
    balanced = []
    for op_id in OPERATIONS:
        df_op   = df_train[df_train["operation"] == f"OP{op_id}"]
        df_good = df_op[df_op["state_encoded"] == 0]
        df_bad  = df_op[df_op["state_encoded"] == 1]
        n       = min(len(df_good), len(df_bad))
        balanced.append(pd.concat([
            df_good.sample(n=n, random_state=42),
            df_bad.sample(n=n,  random_state=42),
        ]))
    df_train = pd.concat(balanced).sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"Balanced training set: {len(df_train):,} windows")
    print(df_train["state_encoded"].value_counts())
else:
    print("Using full, imbalanced training set.")

Balanced training set: 3,148 windows
state_encoded
1    1574
0    1574
Name: count, dtype: int64


## 4. Feature Scaling

In [10]:
META_COLS = ["machine", "operation", "state", "file", "state_encoded", "op_encoded"]

x_train_raw     = df_train.drop(columns=META_COLS)
y_train_anomaly = df_train["state_encoded"]
y_train_op      = df_train["op_encoded"]

x_test_raw      = df_test.drop(columns=META_COLS)
y_test_anomaly  = df_test["state_encoded"]

x_op05_raw      = op05_test_df.drop(columns=META_COLS)
y_op05_anomaly  = op05_test_df["state_encoded"]

x_train_scaled = z_score_scaler(x_train_raw)
x_test_scaled  = z_score_scaler(x_test_raw)
x_op05_scaled  = z_score_scaler(x_op05_raw)

print(f"Train : {x_train_scaled.shape}")
print(f"Test  : {x_test_scaled.shape}")
print(f"OP05  : {x_op05_scaled.shape}")

Train : (3148, 1000)
Test  : (16504, 1000)
OP05  : (622, 1000)


## 5. Instantiate Model

In [11]:
encoder = VibrationFeatureExtractor()
model   = MultiTaskModel(encoder, num_ops=len(OPERATIONS))
print(model)

MultiTaskModel(
  (encoder): VibrationFeatureExtractor(
    (features): Sequential(
      (0): Conv1d(1, 32, kernel_size=(64,), stride=(4,), padding=(32,))
      (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool1d(kernel_size=4, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
      (7): AdaptiveAvgPool1d(output_size=5)
    )
  )
  (anomaly_head): Linear(in_features=320, out_features=2, bias=True)
  (op_head): Linear(in_features=320, out_features=12, bias=True)
)


## 6. Optional Contrastive Pre-Training

When enabled, the shared encoder is first pre-trained with contrastive pairs
built from the combined training set. The learned encoder weights are then
transferred into `MultiTaskModel` before the multi-task fine-tuning step.

In [ ]:
if config["do_contrastive_pretraining"]:
    print(">>> Starting Contrastive Representation Learning Loop...")

    contrastive_encoder = VibrationFeatureExtractor()
    contrastive_model   = ContrastiveModel(contrastive_encoder)

    X1, X2, Y_pairs        = create_balanced_contrastive_pairs(x_train_scaled, y_train_anomaly)
    contrastive_dataloader = DataLoader(
        ContrastiveDataset(X1, X2, Y_pairs),
        batch_size=config["batch_size_contrastive"], shuffle=True
    )

    contrastive_model.set_mode("contrastive")
    optimizer_con = torch.optim.Adam(contrastive_model.parameters(), lr=config["contrastive_lr"])

    for epoch in range(config["contrastive_epochs"]):
        epoch_loss = 0.0
        for x1_b, x2_b, y_b in contrastive_dataloader:
            optimizer_con.zero_grad()
            loss = contrastive_loss(contrastive_model(x1_b), contrastive_model(x2_b), y_b)
            loss.backward()
            optimizer_con.step()
            epoch_loss += loss.item()
        print(f"Contrastive Epoch [{epoch+1}/{config['contrastive_epochs']}] - Loss: {epoch_loss/len(contrastive_dataloader):.4f}")

    model.encoder.load_state_dict(contrastive_encoder.state_dict())
    print(">>> Contrastive encoder weights transferred to MultiTaskModel.")
else:
    print(">>> Skipping Contrastive Pre-Training (disabled in config).")

>>> Starting Contrastive Representation Learning Loop...


## 7. Multi-Task Classification Training

Each batch contributes two cross-entropy losses:
- `L_anomaly` — good or bad machine state?
- `L_op`      — which operation produced this vibration?

Total loss: `L = L_anomaly + aux_weight × L_op`

In [ ]:
train_loader = DataLoader(
    MultiTaskDataset(x_train_scaled, y_train_anomaly, y_train_op),
    batch_size=config["batch_size_classify"], shuffle=True
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config["classification_lr"])

model.train()
print(f">>> Starting Multi-Task Training  (aux_weight={config['aux_weight']}, epochs={config['classification_epochs']})\n")

for epoch in range(config["classification_epochs"]):
    total_loss = anomaly_loss_sum = op_loss_sum = 0.0
    correct = total = 0

    for inputs, y_a, y_o in train_loader:
        optimizer.zero_grad()
        anomaly_logits, op_logits = model(inputs)
        l_anomaly = criterion(anomaly_logits, y_a)
        l_op      = criterion(op_logits,      y_o)
        loss      = l_anomaly + config["aux_weight"] * l_op
        loss.backward()
        optimizer.step()

        n = inputs.size(0)
        total_loss       += loss.item()       * n
        anomaly_loss_sum += l_anomaly.item()  * n
        op_loss_sum      += l_op.item()       * n
        _, predicted = torch.max(anomaly_logits, 1)
        total   += n
        correct += (predicted == y_a).sum().item()

    N = total
    print(
        f"Epoch [{epoch+1:2d}/{config['classification_epochs']}]  "
        f"Total: {total_loss/N:.4f}  "
        f"Anomaly: {anomaly_loss_sum/N:.4f}  "
        f"Op: {op_loss_sum/N:.4f}  "
        f"Anomaly Acc: {correct/total*100:.1f}%"
    )

## 8. Evaluation

### 8a. OP05 Test Set — Direct Comparison with Baseline

Baseline best results (from `results.txt`):

| Config | bad Precision | bad Recall | bad F1 | Macro F1 |
|--------|--------------|------------|--------|----------|
| No contrastive, no downsampling | 1.00 | 0.71 | 0.83 | 0.91 |
| No contrastive, downsampling    | 0.76 | 0.92 | 0.83 | 0.91 |
| Contrastive + downsampling      | 0.61 | 0.89 | 0.72 | 0.85 |

In [ ]:
print("=" * 60)
print("  Anomaly Detection — OP05 Test Set (same as baseline split)")
print("=" * 60)
results_op05 = evaluate_multitask_model(
    model, x_op05_scaled, y_op05_anomaly,
    target_names=["good", "bad"], config=config, export_path="results.txt",
)

### 8b. All Operations Test Set — Generalisation

In [ ]:
print("=" * 60)
print("  Anomaly Detection — All Operations Test Set")
print("=" * 60)
results_all = evaluate_multitask_model(
    model, x_test_scaled, y_test_anomaly,
    target_names=["good", "bad"], config=config, export_path="results.txt",
)

### 8c. Per-Operation Breakdown

In [ ]:
per_op_rows = []
for op_id in OPERATIONS:
    mask  = df_test["operation"] == f"OP{op_id}"
    x_op  = z_score_scaler(df_test[mask].drop(columns=META_COLS))
    y_op  = df_test[mask]["state_encoded"]
    x_t   = torch.tensor(np.asarray(x_op), dtype=torch.float32).unsqueeze(1)
    preds = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(x_t), 64):
            _, p = torch.max(model.predict_anomaly(x_t[start:start+64]), 1)
            preds.extend(p.cpu().numpy())
    y_true   = y_op.to_numpy() if hasattr(y_op, "to_numpy") else np.asarray(y_op)
    macro_f1 = f1_score(y_true, np.array(preds), average="macro",  zero_division=0)
    bad_f1   = f1_score(y_true, np.array(preds), pos_label=1, average="binary", zero_division=0)
    per_op_rows.append({"Operation": f"OP{op_id}", "Macro F1": macro_f1, "Bad F1": bad_f1, "N test": int(mask.sum())})

per_op_df = pd.DataFrame(per_op_rows)
print(per_op_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
x_pos = range(len(per_op_df))
ax.bar([i - 0.2 for i in x_pos], per_op_df["Macro F1"], width=0.4, label="Macro F1")
ax.bar([i + 0.2 for i in x_pos], per_op_df["Bad F1"],   width=0.4, label="Bad (anomaly) F1")
ax.set_xticks(list(x_pos))
ax.set_xticklabels(per_op_df["Operation"], rotation=30)
ax.set_ylim(0, 1.05)
ax.set_ylabel("F1 Score")
ax.set_title("Per-Operation Anomaly Detection — Multi-Task Model")
ax.legend()
plt.tight_layout()
plt.show()

### 8d. Confusion Matrix — OP05 Test Set

In [ ]:
cm = results_op05["confusion_matrix"]
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Predicted Good", "Predicted Bad"],
    yticklabels=["True Good",      "True Bad"],
)
plt.title("Confusion Matrix — OP05 — Multi-Task Model")
plt.tight_layout()
plt.show()